# Silver — Clean, Validate, Deduplicate, Reconcile

Transforms raw Bronze STRING tables into properly typed, trustworthy Delta tables:

- **Type parsing** — `try_cast` to DATE, INT, DECIMAL, TIMESTAMP (safe: unparseable values become NULL, not errors)
- **Business-rule validation matrix** — mandatory fields, valid numeric ranges, allowed statuses, supported currencies, non-future timestamps, and referential integrity (orphan transaction order IDs)
- **Quarantine routing** — every rejected row is routed to a dedicated quarantine table with a human-readable `rejection_reason` (never silently dropped)
- **Deduplication** — deterministic `ROW_NUMBER()` window on business keys (`order_id`, `txn_id`) ordered by `_row_hash` (stable keep-one rule); duplicate losers are tracked separately with `rejection_reason = 'DUPLICATE_*'`
- **Audit preservation** — all Bronze audit identifiers (`_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`) are carried through to every output table
- **Reconciliation** — proves the accounting invariant: Bronze = Accepted Silver + Quarantined + Duplicate Losers, with zero overlap between buckets

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
print(f"Source: {CATALOG}.{SCHEMA}.bronze_*  ->  Target: {CATALOG}.{SCHEMA}.silver_* / quarantine_*")

In [0]:
# --- Silver Orders: parse -> validate -> dedup -> write ---

# 1. Parse: try_cast to proper types, normalize strings, keep ALL audit cols
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orders_parsed AS
SELECT
    trim(NULLIF(order_id, ''))                  AS order_id,
    trim(NULLIF(customer_id, ''))                AS customer_id,
    try_cast(trim(order_date) AS DATE)           AS order_date,
    trim(NULLIF(category, ''))                   AS category,
    try_cast(trim(quantity) AS INT)              AS quantity,
    try_cast(trim(unit_price) AS DECIMAL(18,2))  AS unit_price,
    lower(trim(NULLIF(order_status, '')))        AS order_status,
    upper(trim(NULLIF(country, '')))             AS country,
    _ingested_at, _source_file, _batch_id, _row_hash
FROM `{CATALOG}`.`{SCHEMA}`.`bronze_orders`
""")

# 2. Validate: build rejection_reason from business-rule matrix
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders_validated AS
SELECT *,
    NULLIF(concat_ws(' | ',
        CASE WHEN order_id     IS NULL THEN 'NULL_ORDER_ID' END,
        CASE WHEN customer_id   IS NULL THEN 'NULL_CUSTOMER_ID' END,
        CASE WHEN order_date    IS NULL THEN 'INVALID_ORDER_DATE' END,
        CASE WHEN order_date > current_date() THEN 'FUTURE_ORDER_DATE' END,
        CASE WHEN quantity      IS NULL THEN 'INVALID_QUANTITY' END,
        CASE WHEN quantity     <= 0     THEN 'NONPOSITIVE_QUANTITY' END,
        CASE WHEN unit_price    IS NULL THEN 'INVALID_UNIT_PRICE' END,
        CASE WHEN unit_price   <= 0     THEN 'NONPOSITIVE_UNIT_PRICE' END,
        CASE WHEN order_status  IS NULL THEN 'NULL_ORDER_STATUS' END,
        CASE WHEN order_status NOT IN ('delivered','shipped','placed','cancelled','returned') THEN 'INVALID_ORDER_STATUS' END,
        CASE WHEN category      IS NULL THEN 'NULL_CATEGORY' END,
        CASE WHEN category     NOT IN ('Beauty','Sports','Clothing','Grocery','Books','Home','Electronics') THEN 'INVALID_CATEGORY' END,
        CASE WHEN country        IS NULL THEN 'NULL_COUNTRY' END
    ), '') AS rejection_reason
FROM orders_parsed
""")

# 3. Dedup: keep one row per order_id, deterministic via _row_hash ordering
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders_deduped AS
SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _row_hash) AS _dup_rank
FROM orders_validated WHERE rejection_reason IS NULL
""")

# 4. Write silver_orders (accepted, typed, deduplicated)
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`silver_orders` USING DELTA AS
SELECT order_id, customer_id, order_date, category, quantity, unit_price,
       order_status, country, _ingested_at, _source_file, _batch_id, _row_hash
FROM orders_deduped WHERE _dup_rank = 1
""")

# 5. Write quarantine_orders (validation failures + duplicate losers)
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`quarantine_orders` USING DELTA AS
SELECT order_id, customer_id, order_date, category, quantity, unit_price,
       order_status, country, rejection_reason,
       _ingested_at, _source_file, _batch_id, _row_hash
FROM orders_validated WHERE rejection_reason IS NOT NULL
UNION ALL
SELECT order_id, customer_id, order_date, category, quantity, unit_price,
       order_status, country, 'DUPLICATE_ORDER_ID',
       _ingested_at, _source_file, _batch_id, _row_hash
FROM orders_deduped WHERE _dup_rank > 1
""")

_s = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`silver_orders`").count()
_q = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`quarantine_orders`").count()
print(f"silver_orders:     {_s:,}")
print(f"quarantine_orders: {_q:,}")

In [0]:
# --- Silver Transactions: parse -> validate (incl. ref integrity) -> dedup -> write ---

# 1. Parse: try_cast to proper types, normalize strings, keep ALL audit cols
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_parsed AS
SELECT
    trim(NULLIF(txn_id, ''))                    AS txn_id,
    trim(NULLIF(order_id, ''))                  AS order_id,
    lower(trim(NULLIF(payment_method, '')))     AS payment_method,
    try_cast(trim(amount) AS DECIMAL(18,2))     AS amount,
    upper(trim(NULLIF(currency, '')))           AS currency,
    lower(trim(NULLIF(txn_status, '')))          AS txn_status,
    try_cast(trim(txn_ts) AS TIMESTAMP)         AS txn_ts,
    _ingested_at, _source_file, _batch_id, _row_hash
FROM `{CATALOG}`.`{SCHEMA}`.`bronze_transactions`
""")

# 2. Validate: business-rule matrix + referential integrity (orphan order_id)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_validated AS
SELECT t.*,
    NULLIF(concat_ws(' | ',
        CASE WHEN t.txn_id          IS NULL THEN 'NULL_TXN_ID' END,
        CASE WHEN t.order_id          IS NULL THEN 'NULL_ORDER_ID' END,
        CASE WHEN t.amount           IS NULL THEN 'INVALID_AMOUNT' END,
        CASE WHEN t.amount          <= 0     THEN 'NONPOSITIVE_AMOUNT' END,
        CASE WHEN t.payment_method   IS NULL THEN 'NULL_PAYMENT_METHOD' END,
        CASE WHEN t.payment_method NOT IN ('card','mobile_wallet','bank_transfer','cash','paypal') THEN 'INVALID_PAYMENT_METHOD' END,
        CASE WHEN t.currency         IS NULL THEN 'NULL_CURRENCY' END,
        CASE WHEN t.currency        NOT IN ('USD','EUR','GBP','INR','JPY','CAD','AUD','SGD','BDT') THEN 'UNSUPPORTED_CURRENCY' END,
        CASE WHEN t.txn_status       IS NULL THEN 'NULL_TXN_STATUS' END,
        CASE WHEN t.txn_status      NOT IN ('success','failed','pending','refunded') THEN 'INVALID_TXN_STATUS' END,
        CASE WHEN t.txn_ts           IS NULL THEN 'INVALID_TXN_TS' END,
        CASE WHEN t.txn_ts         > current_timestamp() THEN 'FUTURE_TXN_TS' END,
        CASE WHEN s.order_id          IS NULL AND t.order_id IS NOT NULL THEN 'ORPHAN_ORDER_ID' END
    ), '') AS rejection_reason
FROM txn_parsed t
LEFT JOIN (SELECT DISTINCT order_id FROM `{CATALOG}`.`{SCHEMA}`.`silver_orders`) s
    ON t.order_id = s.order_id
""")

# 3. Dedup: keep one row per txn_id, deterministic via _row_hash ordering
spark.sql("""
CREATE OR REPLACE TEMP VIEW txn_deduped AS
SELECT *, ROW_NUMBER() OVER (PARTITION BY txn_id ORDER BY _row_hash) AS _dup_rank
FROM txn_validated WHERE rejection_reason IS NULL
""")

# 4. Write silver_transactions (accepted, typed, deduplicated)
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`silver_transactions` USING DELTA AS
SELECT txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
       _ingested_at, _source_file, _batch_id, _row_hash
FROM txn_deduped WHERE _dup_rank = 1
""")

# 5. Write quarantine_transactions (validation failures + duplicate losers)
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`quarantine_transactions` USING DELTA AS
SELECT txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
       rejection_reason, _ingested_at, _source_file, _batch_id, _row_hash
FROM txn_validated WHERE rejection_reason IS NOT NULL
UNION ALL
SELECT txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
       'DUPLICATE_TXN_ID', _ingested_at, _source_file, _batch_id, _row_hash
FROM txn_deduped WHERE _dup_rank > 1
""")

_s = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`silver_transactions`").count()
_q = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`quarantine_transactions`").count()
print(f"silver_transactions:     {_s:,}")
print(f"quarantine_transactions: {_q:,}")

In [0]:
from pyspark.sql import functions as F

DATASETS = [
    ("orders", "bronze_orders", "silver_orders", "quarantine_orders", "DUPLICATE_ORDER_ID"),
    ("transactions", "bronze_transactions", "silver_transactions", "quarantine_transactions", "DUPLICATE_TXN_ID"),
]

all_passed = True
print("=" * 70)
print("SILVER RECONCILIATION — ACCOUNTING INVARIANT")
print("  Bronze = Accepted Silver + Quarantined + Duplicate Losers")
print("=" * 70)

for label, bronze_t, silver_t, quar_t, dup_label in DATASETS:
    fqn = lambda t: f"`{CATALOG}`.`{SCHEMA}`.`{t}`"

    bronze_cnt = spark.table(fqn(bronze_t)).count()
    silver_cnt = spark.table(fqn(silver_t)).count()

    quar_df = spark.table(fqn(quar_t))
    quar_cnt = quar_df.count()
    dup_cnt  = quar_df.filter(F.col("rejection_reason") == dup_label).count()
    valfail_cnt = quar_cnt - dup_cnt

    # Accounting invariant: Bronze = Silver + Quarantine (total)
    inv_ok = (bronze_cnt == silver_cnt + quar_cnt)
    # Three-way: Bronze = Silver + Validation failures + Duplicates
    three_ok = (bronze_cnt == silver_cnt + valfail_cnt + dup_cnt)

    # Zero overlap: no _row_hash in both silver and quarantine
    overlap = (spark.table(fqn(silver_t)).select("_row_hash")
               .intersect(quar_df.select("_row_hash")).count())
    overlap_ok = (overlap == 0)

    # Full coverage: distinct _row_hash across all buckets = Bronze count
    total_distinct = (spark.table(fqn(silver_t)).select("_row_hash")
                      .union(quar_df.select("_row_hash")).distinct().count())
    coverage_ok = (total_distinct == bronze_cnt)

    ok = inv_ok and three_ok and overlap_ok and coverage_ok
    all_passed = all_passed and ok

    print(f"\n--- {label} ---")
    print(f"  Bronze rows:               {bronze_cnt:,}")
    print(f"  Accepted Silver:           {silver_cnt:,}")
    print(f"  Quarantine (total):        {quar_cnt:,}")
    print(f"    Validation failures:      {valfail_cnt:,}")
    print(f"    Duplicate losers:        {dup_cnt:,}")
    print(f"  Bronze = Silver + Quarantine:        {'PASS' if inv_ok else 'FAIL'}")
    print(f"  Bronze = Silver + ValFail + Dup:      {'PASS' if three_ok else 'FAIL'}")
    print(f"  Zero overlap (Silver / Quarantine):    {'PASS' if overlap_ok else 'FAIL'} ({overlap} shared)")
    print(f"  Full coverage (distinct _row_hash):    {'PASS' if coverage_ok else 'FAIL'} ({total_distinct:,} = {bronze_cnt:,})")
    print(f"  OVERALL: {'PASS' if ok else 'FAIL'}")

    # Rejection reason breakdown
    print(f"  Rejection reasons:")
    for row in quar_df.groupBy("rejection_reason").count().orderBy("count", ascending=False).collect():
        print(f"    {row['rejection_reason']:45s} {row['count']:>10,}")

print(f"\n{'=' * 70}")
print(f"RESULT: {'ALL CHECKS PASSED' if all_passed else 'FAILURES DETECTED'}")
print("=" * 70)
assert all_passed, "Reconciliation FAILED — see details above."